<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_basic/xor_detail.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings("ignore")

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

# PyTorch 및 환경 정보 출력
print(f"PyTorch 버전: {torch.__version__}")

In [1]:
import numpy as np
# ============================================================
# 활성화 함수 정의
# ============================================================
def sigmoid(x):
    """
    시그모이드 함수: 입력값을 0~1 사이로 변환
    - 큰 양수 → 1에 가까워짐
    - 큰 음수 → 0에 가까워짐
    - 0 → 0.5
    """
    return 1 / (1 + np.exp(-x))

In [ ]:
# ============================================================
# XOR 데이터 준비
# ============================================================
# 입력: 2개의 이진 값 (A, B)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

# 정답: XOR 진리표
# 0 XOR 0 = 0
# 0 XOR 1 = 1
# 1 XOR 0 = 1
# 1 XOR 1 = 0
y = np.array([0, 1, 1, 0])

print("=" * 70)
print("🧠 XOR 문제를 해결하는 2층 신경망")
print("=" * 70)
print("\n📌 XOR 진리표:")
print("   A | B | 출력")
print("  ---|---|-----")
for i in range(4):
    print(f"   {X[i, 0]} | {X[i, 1]} |  {y[i]}")

In [ ]:
# ============================================================
# 네트워크 구조 설명
# ============================================================
print("\n" + "=" * 70)
print("🏗️  네트워크 구조")
print("=" * 70)
print("""
    입력층 (2개) → 은닉층 (2개) → 출력층 (1개)

    입력층: A, B (0 또는 1)
    은닉층: h1, h2 (시그모이드 활성화)
    출력층: output (시그모이드 활성화)
""")

# ============================================================
# 가중치 및 편향 설정
# ============================================================
print("=" * 70)
print("⚙️  가중치 및 편향 설정")
print("=" * 70)

# 1단계: 입력층 → 은닉층
# W1[i,j] = i번째 입력이 j번째 은닉 뉴런에 주는 가중치
W1 = np.array([[20, 20],  # A → h1(가중치 20), A → h2(가중치 20)
               [20, 20]])  # B → h1(가중치 20), B → h2(가중치 20)

# b1[j] = j번째 은닉 뉴런의 편향
b1 = np.array([-30, -10])  # h1의 편향: -30, h2의 편향: -10

print("\n📊 은닉층 가중치 (W1):")
print(f"   A → h1: {W1[0, 0]}, A → h2: {W1[0, 1]}")
print(f"   B → h1: {W1[1, 0]}, B → h2: {W1[1, 1]}")
print(f"\n📊 은닉층 편향 (b1):")
print(f"   h1: {b1[0]}, h2: {b1[1]}")

print("\n💡 은닉층 뉴런의 역할:")
print(f"   h1: z = 20A + 20B - 30 → A=1 AND B=1일 때만 활성화 (AND 게이트)")
print(f"   h2: z = 20A + 20B - 10 → A=1 OR B=1이면 활성화 (OR 게이트)")

# 2단계: 은닉층 → 출력층
# W2[i] = i번째 은닉 뉴런이 출력에 주는 가중치
W2 = np.array([20, -20])  # h1 → output(가중치 20), h2 → output(가중치 -20)

# b2 = 출력 뉴런의 편향
b2 = -10

print(f"\n📊 출력층 가중치 (W2):")
print(f"   h1 → output: {W2[0]}")
print(f"   h2 → output: {W2[1]}")
print(f"\n📊 출력층 편향 (b2): {b2}")

print("\n💡 출력층의 역할:")
print(f"   z = 20h1 - 20h2 - 10")
print(f"   h1이 ON이고 h2가 OFF면 큰 양수 → XOR 불가능한 경우 제거")

# ============================================================
# 순전파 (Forward Propagation) - 각 입력에 대해 계산
# ============================================================
print("\n" + "=" * 70)
print("🚀 순전파 (Forward Propagation) - 상세 계산")
print("=" * 70)

for i in range(4):
    print("\n" + "-" * 70)
    print(f"🔢 테스트 케이스 {i + 1}: A={X[i, 0]}, B={X[i, 1]} (정답: {y[i]})")
    print("-" * 70)

    # --------------------------------------------------------
    # Step 1: 은닉층 입력 계산 (가중합)
    # --------------------------------------------------------
    print("\n[Step 1] 은닉층 입력 계산")

    # z1 = X @ W1 + b1
    # 각 은닉 뉴런의 가중합 계산
    z1_h1 = X[i, 0] * W1[0, 0] + X[i, 1] * W1[1, 0] + b1[0]  # h1의 입력
    z1_h2 = X[i, 0] * W1[0, 1] + X[i, 1] * W1[1, 1] + b1[1]  # h2의 입력

    z1 = np.array([z1_h1, z1_h2])

    print(f"   h1 입력 (z1[0]) = {X[i, 0]}×{W1[0, 0]} + {X[i, 1]}×{W1[1, 0]} + {b1[0]}")
    print(f"                    = {X[i, 0] * W1[0, 0]} + {X[i, 1] * W1[1, 0]} + {b1[0]}")
    print(f"                    = {z1_h1}")

    print(f"   h2 입력 (z1[1]) = {X[i, 0]}×{W1[0, 1]} + {X[i, 1]}×{W1[1, 1]} + {b1[1]}")
    print(f"                    = {X[i, 0] * W1[0, 1]} + {X[i, 1] * W1[1, 1]} + {b1[1]}")
    print(f"                    = {z1_h2}")

In [ ]:
# --------------------------------------------------------
# Step 2: 은닉층 활성화 (시그모이드 적용)
# --------------------------------------------------------
print("\n[Step 2] 은닉층 활성화 (Sigmoid)")

h1 = sigmoid(z1_h1)
h2 = sigmoid(z1_h2)
h = np.array([h1, h2])

print(f"   h1 = σ({z1_h1}) = {h1:.6f} ≈ {h1:.1f}")
print(f"   h2 = σ({z1_h2}) = {h2:.6f} ≈ {h2:.1f}")

# 뉴런 상태 시각화
h1_status = "🟢 ON" if h1 > 0.5 else "🔴 OFF"
h2_status = "🟢 ON" if h2 > 0.5 else "🔴 OFF"
print(f"\n   은닉층 상태: h1 {h1_status}, h2 {h2_status}")

In [ ]:
# --------------------------------------------------------
# Step 3: 출력층 입력 계산
# --------------------------------------------------------
print("\n[Step 3] 출력층 입력 계산")

z2 = h1 * W2[0] + h2 * W2[1] + b2

print(f"   z_output = {h1:.6f}×{W2[0]} + {h2:.6f}×{W2[1]} + {b2}")
print(f"            = {h1 * W2[0]:.6f} + {h2 * W2[1]:.6f} + {b2}")
print(f"            = {z2:.6f}")

In [ ]:
# --------------------------------------------------------
# Step 4: 최종 출력 (시그모이드 적용)
# --------------------------------------------------------
print("\n[Step 4] 최종 출력 (Sigmoid)")

output = sigmoid(z2)
prediction = int(output > 0.5)

print(f"   output = σ({z2:.6f}) = {output:.6f}")
print(f"   예측값 = {prediction} (threshold: 0.5)")

In [ ]:
# --------------------------------------------------------
# 결과 확인
# --------------------------------------------------------
is_correct = prediction == y[i]
result_emoji = "✅" if is_correct else "❌"